# 1 · Data ingestion, ETL & SQL | 45 minutes
**Business mission:** clean a small sample of real Kaggle customer-support tickets and load them into your **team-owned Supabase tables**. Then answer business questions with SQL.

Source: [Customer Support Ticket Dataset (Kaggle)](https://www.kaggle.com/datasets/suraj520/customer-support-ticket-dataset). **Instructor already downloaded + ingested it**; no Kaggle account or local CSV required.

> A source caveat: `First Response Time` and `Time to Resolution` are timestamps. Their difference is **time from first response to resolution**, not total ticket lifetime or a contractual SLA. Missing durations stay null. Dataset is synthetic and may have noisy text.

## Why this notebook matters
Raw data is almost never ready to analyse: values have stray spaces, inconsistent capitalisation, missing fields, duplicates. If we build dashboards (Notebook 2) or an AI assistant (Notebook 3) on top of messy data, we get wrong numbers and wrong answers. So the very first job of a data team is to **get the data clean and into a database that everyone can query**. That job is called **ETL**.

## The big picture: ETL = Extract, Transform, Load
![ETL pipeline: Extract reads raw rows, Transform cleans them in pandas, Load writes them to your team table, Analyse answers questions with SQL](https://raw.githubusercontent.com/Ngxba/fpt-ghc-data-ai-workshop-level1/main/assets/nb1_etl_pipeline.png)
Each section below is one of these steps. For every step you will see **what we do**, **why we do it** and **what you should see** when it works. Five cells need your code: `TODO 1` to `TODO 5`; everything else is provided.

## Where the data lives
![The Supabase database with a read-only shared schema (raw_tickets, ticket_types, knowledge_articles) and your writable team_XX schema (clean_tickets, knowledge_articles); the ETL arrow goes from raw_tickets to clean_tickets](https://raw.githubusercontent.com/Ngxba/fpt-ghc-data-ai-workshop-level1/main/assets/nb1_data_map.png)

## Vocabulary you will meet
| Term | Meaning |
|---|---|
| **Schema** | A named folder of tables inside one database (`shared`, `team_01`...). |
| **Primary key (PK)** | Column(s) whose value is **unique per row** and never empty: it identifies a row. |
| **Foreign key (FK)** | A column that must match a primary key in **another table**; the database rejects values that do not exist there. |
| **NULL** | "Unknown / missing". Not the same as `0` or an empty string. |
| **DataFrame** | pandas' in-memory table: rows and typed columns. |
| **UPSERT** | `INSERT`, or `UPDATE` if the row already exists, so re-running is safe. |
| **JOIN** | Combine rows of two tables where a column matches. |
| **Aggregation** | Collapse many rows into one number per group: `COUNT`, `AVG`, ... with `GROUP BY`. |

## Step 0 · Setup
**What:** install the Python packages this lab uses.
- `psycopg` = the driver that lets Python talk to PostgreSQL.
- `pandas` = tables (DataFrames) in Python, our cleaning tool.
- `plotly` = charts (Notebook 2). `openai` = the AI API (Notebook 3).
- `python-dotenv` = reads secrets from a local `.env` file (explained below).

**Why one cell:** every notebook installs what it needs, so each one can run on its own in Google Colab. The `-q` flag keeps the output quiet. **What you should see:** nothing, or a few install lines, no errors.

In [1]:
!pip install -q "psycopg[binary]>=3.2,<4" "pandas>=2.2,<3" "plotly>=5,<7" "openai>=1.55,<3"

In [2]:
!pip install -q python-dotenv

## Step 0 · Colab secrets (Colab only)
**What:** on Google Colab, load your credentials from **Secrets** (the key icon in the left sidebar), so you enter them once for all notebooks instead of typing them in each one. For every name below click **Add new secret**, paste the value, and switch **Notebook access** on. On local Jupyter skip this: your `.env` file is used.

| Secret name | Value |
|---|---|
| `LAB_PROJECT_REF` | Supabase project ref, given by your instructor |
| `LAB_POOLER_HOST` | Transaction pooler host (host name only, not the full URL) |
| `LAB_TEAM_NUMBER` | Your team number, e.g. `1` |
| `LAB_DB_PASSWORD` | Your team database password |

**What you should see:** nothing printed. If a secret is missing, the cell lists it and the next cells ask you to type it.

In [ ]:
# Colab only: copy your Colab Secrets into environment variables.
# Outside Colab (local Jupyter) this cell does nothing and your .env file / the prompts are used.
import os
try:
    from google.colab import userdata
except ImportError:
    userdata = None

if userdata is not None:
    missing = []
    for name in ["LAB_PROJECT_REF", "LAB_POOLER_HOST", "LAB_TEAM_NUMBER", "LAB_DB_PASSWORD"]:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:  # secret not created, or "Notebook access" is switched off
            missing.append(name)
    if missing:
        print("Not found in Colab Secrets (you will be asked to type them):", ", ".join(missing))

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

## Step 0 · Connect (provided helper; only your team tables are writable)
**What:** this cell reads your team credentials, then defines three small helper functions that we use for the rest of the lab.

**How it works**
1. **Credentials:** `os.getenv(...)` first looks for a value in the environment, which `load_dotenv()` fills from a `.env` file. If a value is missing, the cell asks you with `input(...)`, or `getpass` for the password (it hides what you type). **Never write a password inside the notebook**: notebooks get shared and committed.
2. **Your schema:** the team number becomes your private schema name, e.g. team `1` → `team_01`, and your database user `lab_team_01.<project>`. That user is only allowed to write in *your* schema.
3. **`connect()`** opens a connection to Supabase through its *transaction pooler* (port `6543`), which shares a limited number of database connections between many students. SSL is required, and prepared statements are switched off because the pooler does not support them.
4. **`fetch_df(query)`** runs a `SELECT` and returns the result as a pandas **DataFrame**.
5. **`run_sql(query)`** runs a statement that changes data (no result). It runs inside a **transaction**: either everything succeeds, or nothing is saved.

Both helpers open a connection, use it and close it again (`closing(...)`), so you never leave connections open.

In [4]:
import os, getpass
from contextlib import closing
import psycopg
from psycopg import sql
import pandas as pd

# Reenter these in EACH notebook. Use the team credential supplied privately by your instructor.
PROJECT_REF = os.getenv("LAB_PROJECT_REF") or input("Supabase project ref: ").strip()
POOLER_HOST = os.getenv("LAB_POOLER_HOST") or input("Transaction pooler host (not full URL): ").strip()
TEAM_NUMBER = int(os.getenv("LAB_TEAM_NUMBER") or input("Team number (e.g. 1): ").strip())
TEAM_SCHEMA = f"team_{TEAM_NUMBER:02d}"
DB_PASSWORD = os.getenv("LAB_DB_PASSWORD") or getpass.getpass("Team database password: ")
DB_USER = f"lab_team_{TEAM_NUMBER:02d}.{PROJECT_REF}"

def connect():
    # Supabase Free transaction pooler; psycopg prepared statements disabled.
    return psycopg.connect(host=POOLER_HOST, port=6543, dbname="postgres",
                           user=DB_USER, password=DB_PASSWORD, sslmode="require",
                           connect_timeout=12, prepare_threshold=None)

def fetch_df(query, params=None):
    with closing(connect()) as conn:
        with conn.cursor() as cur:
            cur.execute(query, params)
            if cur.description is None: return pd.DataFrame()
            return pd.DataFrame(cur.fetchall(), columns=[x.name for x in cur.description])

def run_sql(query, params=None):
    with closing(connect()) as conn:
        with conn.transaction():
            with conn.cursor() as cur:
                cur.execute(query, params)

print("Connected to:", TEAM_SCHEMA)
print(fetch_df("SELECT current_user AS user_name, current_database() AS db_name"))

Connected to: team_01
     user_name   db_name
0  lab_team_01  postgres


**What you should see:** `Connected to: team_XX` and a small table with your database user (`lab_team_XX`) and the database name `postgres`. If you get an error here, nothing else will work: check the team number and password with your instructor.

## Step 1 · Explore the schema before changing data
**What:** ask the database to describe its own tables. Every PostgreSQL database has a built-in catalogue called `information_schema`; `information_schema.columns` lists every column of every table with its data type.

**Why:** a data engineer never loads data blindly. First we learn the **shape** of the source (`shared.raw_tickets`) and of the **destination** (`team_XX.clean_tickets`), because the destination has rules: required columns, a primary key, and a foreign key to the list of valid ticket types.

**How to read the output:** one row per column: `table_schema`, `table_name`, `column_name`, `data_type`. Compare the raw table with the clean table: which columns exist in both, and which type does each have?

You can also read `reference/schema_reference.sql` in the repo, which holds the table definitions.

**Exercise:** Name one primary key, one foreign key, and one reason the team has its own schema. The instructor has already created the tables—do not run DDL.

*Hints: a primary key identifies a row; a foreign key says `REFERENCES` another table; think about what could go wrong if all teams wrote into the same table.*

In [5]:
display(fetch_df("""SELECT table_schema, table_name, column_name, data_type
  FROM information_schema.columns
  WHERE (table_schema='shared' AND table_name IN ('raw_tickets','ticket_types','knowledge_articles'))
     OR (table_schema=%s AND table_name IN ('clean_tickets','knowledge_articles'))
  ORDER BY table_schema, table_name, ordinal_position""",(TEAM_SCHEMA,)).head(45))

,table_schema,table_name,column_name,data_type
0,shared,knowledge_articles,article_id,text
1,shared,knowledge_articles,title,text
2,shared,knowledge_articles,category,text
3,shared,knowledge_articles,content,text
4,shared,knowledge_articles,embedding,USER-DEFINED
5,shared,raw_tickets,ticket_id,integer
6,shared,raw_tickets,ticket_type,text
7,shared,raw_tickets,ticket_subject,text
8,shared,raw_tickets,ticket_description,text
9,shared,raw_tickets,status,text


**What you should see:** 34 rows of column names, one per column. Note that `clean_tickets` has an extra column, `response_to_resolution_hours`, which does not exist in the raw table. We will **derive** it in Step 3.

## Step 2 · Extract: read the raw tickets
**What:** copy a sample of the raw data from the database into a pandas DataFrame called `raw`.

**How:** `SELECT * FROM shared.raw_tickets ORDER BY ticket_id LIMIT 600`
- `SELECT *` = all columns, `FROM` = which table.
- `ORDER BY ticket_id` = a fixed order. Without it, `LIMIT` may return a *different* 600 rows each time, and your results would not match your neighbour's.
- `LIMIT 600` = take only the first 600 rows. A small sample keeps the workshop fast.

Then we **profile** the data before touching it: `raw.head(3)` shows 3 example rows, and `raw.isna().sum()` counts the missing values in every column. **Always look before you clean.**

In [6]:
raw = fetch_df("SELECT * FROM shared.raw_tickets ORDER BY ticket_id LIMIT 600")
print("Raw records:",len(raw)); display(raw.head(3))
print("Null counts:"); display(raw.isna().sum().to_frame('nulls').T)

Raw records: 600


,ticket_id,ticket_type,ticket_subject,ticket_description,status,priority,channel,product,first_response_at,resolved_at,satisfaction_rating,resolution_text
0,1,Technical issue,Product setup,I'm having an issue with the [product]. Please...,Pending Customer Response,Critical,Social media,GoPro Hero,2023-06-01 12:15:36+00:00,NaT,None,None
1,2,Technical issue,Peripheral compatibility,I'm having an issue with the [product]. Please...,Pending Customer Response,Critical,Chat,LG Smart TV,2023-06-01 16:45:38+00:00,NaT,None,None
2,3,Technical issue,Network problem,I'm facing a problem with my [product]. The [p...,Closed,Low,Social media,Dell XPS,2023-06-01 11:14:38+00:00,2023-06-01 18:05:38+00:00,3.00,Case maybe show recently my computer follow.


Null counts:


,ticket_id,ticket_type,ticket_subject,ticket_description,status,priority,channel,product,first_response_at,resolved_at,satisfaction_rating,resolution_text
nulls,0,0,0,0,0,0,0,0,191,392,392,392


**What you should see:** `Raw records: 600`, three example tickets, and a row of null counts.

**How to read the null counts:** in the reference run, `ticket_id`, `ticket_type` and `status` have **no** nulls, but `resolved_at`, `satisfaction_rating` and `resolution_text` are missing for about two thirds of the tickets (392 of 600), and `first_response_at` for 191. That is not an error: tickets that are still open or pending have no resolution yet. **A missing value can be real information**, so in Step 3 we must not "fix" it by filling in zeros.

## Step 3 · Transform: clean the tickets in pandas
**What:** turn `raw` into a trustworthy DataFrame `df` in five small steps. We work on a **copy** (`raw.copy()`), so if you break something you can simply re-run from here without querying the database again.

| Step | What we fix | Why it matters |
|---|---|---|
| 3.1 | Text and number types | `" Closed"`, `"closed"` and `"CLOSED"` must count as one status |
| 3.2 | Missing keys and duplicates (**TODO 1, 2**) | A ticket without an ID or type is unusable; duplicates inflate every count |
| 3.3 | Dates and ratings | Text dates cannot be subtracted; bad values must become "missing", not crash |
| 3.4 | A new column: hours to resolution (**TODO 3**) | A business metric we derive from two timestamps |
| 3.5 | pandas missing → database `NULL` | The database must receive `NULL`, not pandas' `NaN` |

**Rule for this whole step:** never invent data. Missing stays missing.

![Raw tickets with stray spaces, wrong capitalisation, a duplicate ID and a missing ID become two clean rows](https://raw.githubusercontent.com/Ngxba/fpt-ghc-data-ai-workshop-level1/main/assets/nb1_transform_before_after.png)

### 3.1 · Normalise types and text

In [7]:
df = raw.copy()    # work on a copy: `raw` stays untouched
df["ticket_id"] = pd.to_numeric(df["ticket_id"], errors="coerce")
df["ticket_type"] = df["ticket_type"].astype("string").str.strip()
df["status"] = df["status"].astype("string").str.strip().str.title()

print("Statuses after normalising:", sorted(df["status"].dropna().unique()))
print("Ticket types:", sorted(df["ticket_type"].dropna().unique()))

Statuses after normalising: ['Closed', 'Open', 'Pending Customer Response']
Ticket types: ['Billing inquiry', 'Cancellation request', 'Product inquiry', 'Refund request', 'Technical issue']


**What each line does**
- `pd.to_numeric(..., errors="coerce")`: turns IDs into numbers; anything that is not a number becomes missing (`NaN`) instead of crashing.
- `.astype("string")`: pandas' text type, which handles missing values cleanly.
- `.str.strip()`: removes leading and trailing spaces.
- `.str.title()`: `"CLOSED"` and `"closed"` both become `"Closed"`. **Remember the exact spelling `Closed`: you need it in TODO 5.**

**What you should see:** a short list of statuses, each spelled one way only, and the five ticket types.

### 3.2 · TODO 1 and TODO 2 · Drop unusable rows and duplicates
- **TODO 1:** `dropna(subset=[...])` removes a row if any of the **listed** columns is missing. Which two columns must never be empty? (The ticket's identifier, and its type. Use the exact column names.)
- **TODO 2:** `drop_duplicates(subset=[...], keep="last")` keeps one row for each distinct value of the listed column(s). If the same ticket appears twice, we keep the **last** copy, usually the most recent update. Which column identifies one ticket?

In [8]:
rows_before = len(df)
# TODO 1: Remove rows with missing ticket ID or ticket type
df = df.dropna(subset=["ticket_id", "ticket_type"])
# TODO 2: Keep one row for each ticket ID
df = df.drop_duplicates(subset=["ticket_id"], keep="last")
print(f"Rows: {rows_before} -> {len(df)}  (removed {rows_before - len(df)})")

Rows: 600 -> 600  (removed 0)


**What you should see:** `Rows: 600 -> 600 (removed 0)` on this sample. That is fine: the instructor's table already has a primary key, so duplicates are unlikely here. We still apply the rule, because **real extracts (CSV exports, API dumps) often contain duplicates**, and the check costs nothing.

### 3.3 · Parse dates and ratings
Timestamps arrive as text. To subtract them we need real date objects.
- `pd.to_datetime(..., utc=True, errors="coerce")`: converts text to timezone-aware timestamps (all in UTC, so no daylight-saving surprises); unreadable values become `NaT` ("not a time"), pandas' missing value for dates.
- `pd.to_numeric(..., errors="coerce")` for `satisfaction_rating` does the same for numbers.

In [9]:
for col in ("first_response_at", "resolved_at"):
    df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")
df["satisfaction_rating"] = pd.to_numeric(df["satisfaction_rating"], errors="coerce")
print(df[["first_response_at", "resolved_at", "satisfaction_rating"]].dtypes)

first_response_at      datetime64[ns, UTC]
resolved_at            datetime64[ns, UTC]
satisfaction_rating                float64
dtype: object


**What you should see:** the first two columns are now `datetime64[..., UTC]` and the rating is a number type, not `object` (text).

### 3.4 · TODO 3 · Derive a new column: hours from first response to resolution
**What:** `response_to_resolution_hours = (resolved_at − first_response_at)` converted to hours. This is a classic *feature engineering* step: a new column that answers a business question and does not exist in the raw data.

**How it works**
1. Subtracting two timestamp columns gives a *duration*; `.dt.total_seconds() / 3600` converts it to hours.
2. `hours.where(condition)` **keeps** the value where the condition is true and sets it to missing everywhere else. We only keep a value when the ticket is `Closed` **and** the result is between 0 and one year (`24*365` hours). Negative or absurd durations are data errors, so they become missing.
3. `.round(2)` keeps two decimals.

![Three tickets on a timeline: a closed ticket gives 6.85 hours, a pending ticket and a ticket with bad data give NULL](https://raw.githubusercontent.com/Ngxba/fpt-ghc-data-ai-workshop-level1/main/assets/nb1_hours_timeline.png)

**TODO 3:** fill in the two timestamp columns, in the right order: later time minus earlier time. (Reminder from the intro: it is *first response → resolution*, not the whole ticket lifetime.)

**Why missing, not zero?** If we filled "unknown" with `0`, every average would be dragged down and the dashboard would lie. Missing values are skipped by `AVG`, which is what we want.

In [10]:
# TODO 3: Compute hours from first response to resolution (not total ticket lifetime)
hours = (df["resolved_at"] - df["first_response_at"]).dt.total_seconds() / 3600
df["response_to_resolution_hours"] = hours.where(
    df["status"].eq("Closed") & hours.between(0, 24*365)
).round(2)

print("Tickets with a duration:", int(df["response_to_resolution_hours"].notna().sum()), "of", len(df))
display(df[["ticket_id", "status", "first_response_at", "resolved_at", "response_to_resolution_hours"]].head(5))

Tickets with a duration: 105 of 600


,ticket_id,status,first_response_at,resolved_at,response_to_resolution_hours
0,1,Pending Customer Response,2023-06-01 12:15:36+00:00,NaT,NaN
1,2,Pending Customer Response,2023-06-01 16:45:38+00:00,NaT,NaN
2,3,Closed,2023-06-01 11:14:38+00:00,2023-06-01 18:05:38+00:00,6.85
3,4,Closed,2023-06-01 07:29:40+00:00,2023-06-01 01:57:40+00:00,NaN
4,5,Closed,2023-06-01 00:12:42+00:00,2023-06-01 19:53:42+00:00,19.68


**What you should see:** a count of tickets that have a duration (only part of the 600) and 5 example rows. A row with `Closed` status, both timestamps and a positive result shows hours (e.g. `6.85`); every other row shows `NaN`. Check one by hand: does `resolved_at − first_response_at` match?

### 3.5 · Convert pandas "missing" into database `NULL`
pandas marks missing values as `NaN` / `NaT` / `<NA>`. The database driver needs the Python value `None` to write a real SQL `NULL`: a `NaT` would raise an error, and a `NaN` could be stored as the *value* "NaN" instead of "unknown", which would break averages. The line below converts every missing value to `None`.

In [11]:
df = df.astype(object).where(pd.notna(df), None)
print("Cleaned tickets:", len(df))
display(df[["ticket_id", "ticket_type", "status", "response_to_resolution_hours"]].head())

Cleaned tickets: 600


,ticket_id,ticket_type,status,response_to_resolution_hours
0,1,Technical issue,Pending Customer Response,None
1,2,Technical issue,Pending Customer Response,None
2,3,Technical issue,Closed,6.85
3,4,Billing inquiry,Closed,None
4,5,Billing inquiry,Closed,19.68


**What you should see:** `Cleaned tickets: 600` and a table where unknown durations now show `None`. **Checkpoint of the Transform step:** you have a clean DataFrame `df`. Nothing has been written to the database yet, so you can still experiment safely.

## Step 4 · Load into your team table: prewritten UPSERT
**What:** write the clean rows from `df` into `team_XX.clean_tickets` in the database. This cell is provided; read it to understand it.

![Flow chart of the loader: skip rows with an unknown ticket type, otherwise INSERT a new ticket or UPDATE an existing one, ending with exactly one row per ticket](https://raw.githubusercontent.com/Ngxba/fpt-ghc-data-ai-workshop-level1/main/assets/nb1_upsert_flow.png)

**What the code does, in order**
1. Choose the columns to write, and fetch the list of **valid ticket types** from `shared.ticket_types`.
2. Keep only rows whose type is in that list. The table has a **foreign key** to `ticket_types`, so the database would reject any other value and fail the whole load.
3. `pgval()` converts each value into something the database driver accepts: pandas timestamps become Python `datetime`, missing values become `None` (= `NULL`).
4. Build the SQL **UPSERT**: `INSERT ... ON CONFLICT (ticket_id) DO UPDATE SET ...`. A new ticket is inserted; a ticket whose ID already exists is **updated**. That is why you can run the cell twice and still end up with exactly one row per ticket (**idempotent**).
5. The values are passed separately through `%s` placeholders. The database inserts them safely, which protects against **SQL injection** and handles quoting and types for us. (The schema name in the f-string is safe: it is built from your numeric team number only.)
6. `executemany` sends all rows inside one **transaction**: all rows are saved, or, if anything fails, none.
7. The last line checks that the table now holds at least one row.

In [12]:
# Provided loader. Idempotent UPSERT makes reruns safe.
from datetime import datetime
columns = ["ticket_id","ticket_type","ticket_subject","status","priority","channel", "product",
           "first_response_at","resolved_at","satisfaction_rating","response_to_resolution_hours"]
allowed_types = set(fetch_df("SELECT ticket_type FROM shared.ticket_types")["ticket_type"])
ready = df[columns].copy()
ready = ready[ready["ticket_type"].isin(allowed_types)]
ready["ticket_id"] = ready["ticket_id"].astype(int)

def pgval(v):
    if v is None or (not isinstance(v, (list,tuple,dict)) and pd.isna(v)): return None
    if hasattr(v,"to_pydatetime"): return v.to_pydatetime()
    return v

placeholders=','.join(['%s']*len(columns))
updates=','.join(f"{c}=EXCLUDED.{c}" for c in columns if c != "ticket_id")
query=f"""INSERT INTO {TEAM_SCHEMA}.clean_tickets ({','.join(columns)})
VALUES ({placeholders}) ON CONFLICT(ticket_id) DO UPDATE SET {updates}"""
with closing(connect()) as conn:
    with conn.transaction():
        with conn.cursor() as cur:
            cur.executemany(query,[tuple(pgval(v) for v in row) for row in ready.itertuples(index=False,name=None)])
print("Loaded:",len(ready),"team rows")
assert int(fetch_df(f"SELECT COUNT(*) AS n FROM {TEAM_SCHEMA}.clean_tickets").iloc[0,0]) >= 1

Loaded: 600 team rows


**What you should see:** `Loaded: 600 team rows`. Let's verify by reading back **from the database** (not from `df`) to prove the data really landed.

In [13]:
display(fetch_df(f"SELECT * FROM {TEAM_SCHEMA}.clean_tickets ORDER BY ticket_id LIMIT 5"))
print("Rows in your team table:", int(fetch_df(f"SELECT COUNT(*) AS n FROM {TEAM_SCHEMA}.clean_tickets").iloc[0, 0]))

,ticket_id,ticket_type,ticket_subject,status,priority,channel,product,first_response_at,resolved_at,satisfaction_rating,response_to_resolution_hours
0,1,Technical issue,Product setup,Pending Customer Response,Critical,Social media,GoPro Hero,2023-06-01 12:15:36+00:00,NaT,None,None
1,2,Technical issue,Peripheral compatibility,Pending Customer Response,Critical,Chat,LG Smart TV,2023-06-01 16:45:38+00:00,NaT,None,None
2,3,Technical issue,Network problem,Closed,Low,Social media,Dell XPS,2023-06-01 11:14:38+00:00,2023-06-01 18:05:38+00:00,3.00,6.85
3,4,Billing inquiry,Account access,Closed,Low,Social media,Microsoft Office,2023-06-01 07:29:40+00:00,2023-06-01 01:57:40+00:00,3.00,None
4,5,Billing inquiry,Data loss,Closed,Low,Email,Autodesk AutoCAD,2023-06-01 00:12:42+00:00,2023-06-01 19:53:42+00:00,1.00,19.68


Rows in your team table: 600


**What you should see:** 5 rows, with `NULL`s shown as `None`/`NaN`, and a row count equal to what you loaded. The data is now **permanent and shared**: Notebooks 2 to 5 read this table.

## Step 5 · Analyse with SQL
Now that the data is clean and in the database, we let the database do the counting. Three ideas are enough for this notebook:
- **`GROUP BY`** splits the rows into groups (one per status, per ticket type...).
- **Aggregate functions** compute one number per group: `COUNT(*)` counts rows, `AVG(x)` averages `x`. **`AVG` ignores `NULL`s**, which is why leaving missing durations as `NULL` in Step 3 matters.
- **`JOIN`** combines two tables on matching columns.

![Tickets are joined with ticket_types, grouped by ticket type, then counted and averaged per group; AVG skips NULL values](https://raw.githubusercontent.com/Ngxba/fpt-ghc-data-ai-workshop-level1/main/assets/nb1_join_groupby.png)

### 5.1 · Warm-up (provided): how many tickets per status?

In [14]:
display(fetch_df(f"""
SELECT status, COUNT(*) AS tickets
FROM {TEAM_SCHEMA}.clean_tickets
GROUP BY status
ORDER BY tickets DESC
"""))

,status,tickets
0,Closed,208
1,Pending Customer Response,201
2,Open,191


**How to read it:** SQL runs in this order: `FROM` (pick the table) → `GROUP BY` (make one group per status) → `SELECT` (compute `COUNT(*)` for each group) → `ORDER BY` (biggest first). **What you should see:** one row per status, and the counts add up to your total number of rows.

### 5.2 · TODO 4 · Business SQL: JOIN and aggregation
**Business question:** *for each ticket type, how many tickets do we have and how long does it take from first response to resolution, on average?*

The query reads `clean_tickets` (alias `t`) and joins the lookup table `ticket_types` (alias `c`). Fill in the two blanks:
- `JOIN ... ON __________`: the condition that links the two tables, i.e. the column they **share** (`t.<column> = c.<column>`).
- `GROUP BY __________`: the column that defines the groups. Every non-aggregated column in `SELECT` must appear in `GROUP BY`.

Note `AVG(t.response_to_resolution_hours)`: it only averages tickets that **have** a duration; tickets with `NULL` are skipped.

In [15]:
# TODO 4: Complete JOIN and GROUP BY (no instructor DDL needed).
q = f"""
SELECT t.ticket_type, COUNT(*) AS total_tickets,
       ROUND(AVG(t.response_to_resolution_hours), 2) AS avg_response_to_resolution_hours
FROM {TEAM_SCHEMA}.clean_tickets t
JOIN shared.ticket_types c ON t.ticket_type = c.ticket_type
GROUP BY t.ticket_type
ORDER BY total_tickets DESC
"""
display(fetch_df(q))

,ticket_type,total_tickets,avg_response_to_resolution_hours
0,Technical issue,132,7.26
1,Cancellation request,124,7.79
2,Refund request,123,9.14
3,Billing inquiry,112,8.45
4,Product inquiry,109,6.15


**What you should see:** one row per ticket type (five of them), sorted by the number of tickets, with an average duration of roughly 6 to 9 hours in the reference run. The `total_tickets` numbers add up to the rows you loaded. If a ticket type is missing, check your JOIN condition.

### 5.3 · TODO 5 · Closed rate (a business metric, not official SLA)
**Business question:** *what share of tickets of each type is already closed?*

The tool is a **conditional aggregate**: `COUNT(*) FILTER (WHERE condition)` counts only the rows that match the condition, while the plain `COUNT(*)` still counts all rows, in the same query.

![Closed rate per ticket type: COUNT(*) counts all tickets, COUNT(*) FILTER counts only the closed ones, and the ratio gives the percentage](https://raw.githubusercontent.com/Ngxba/fpt-ghc-data-ai-workshop-level1/main/assets/nb1_closed_rate.png)
- Fill in the status text. It must match the stored value **exactly, including capital letters**: remember what Step 3.1 did to the status.
- `100.0 * closed / total` gives a percentage. The `100.0` (with a decimal point) forces decimal division instead of integer division.
- `NULLIF(COUNT(*), 0)` turns a zero total into `NULL`, so we never divide by zero.
- `ROUND(..., 1)` keeps one decimal.

Remember the caveat from the top: this is a **closed rate** for our sample, not an official SLA.

In [16]:
# TODO 5: Use FILTER to count closed tickets and calculate closed rate.
q = f"""
SELECT ticket_type,
       COUNT(*) AS total,
       COUNT(*) FILTER (WHERE status = 'Closed') AS closed,
       ROUND(100.0 * COUNT(*) FILTER (WHERE status = 'Closed') / NULLIF(COUNT(*),0), 1) AS closed_pct
FROM {TEAM_SCHEMA}.clean_tickets
GROUP BY ticket_type ORDER BY total DESC
"""
display(fetch_df(q))

,ticket_type,total,closed,closed_pct
0,Technical issue,132,49,37.1
1,Cancellation request,124,40,32.3
2,Refund request,123,46,37.4
3,Billing inquiry,112,39,34.8
4,Product inquiry,109,34,31.2


**What you should see:** per ticket type the number of tickets, how many are `Closed`, and the percentage: around a third in the reference run. Sanity check: `closed` is never larger than `total`.

### Checkpoint
You should see results for each ticket type, and the team-owned table should contain cleaned rows. If something fails, ask the instructor to share a solution cell and re-run—the subsequent notebooks read this same table.

### What you did, step by step
| Step | Action | Tool | Result |
|---|---|---|---|
| 0 | Install and connect | `pip`, `psycopg`, `.env` | `fetch_df` / `run_sql` helpers |
| 1 | Explore the schema | `information_schema` | You know PK, FK and why teams have separate schemas |
| 2 | **Extract** | `SELECT ... LIMIT` | DataFrame `raw` with 600 messy tickets |
| 3 | **Transform** | pandas | DataFrame `df` with clean types, no unusable rows, a new metric |
| 4 | **Load** | SQL UPSERT | Rows saved in `team_XX.clean_tickets` |
| 5 | **Analyse** | `GROUP BY`, `JOIN`, `FILTER` | Business answers per ticket type |

### Discuss with your team
1. Why do we clean the data **before** loading it, and not inside every dashboard?
2. Why is a missing `resolved_at` left as `NULL` instead of `0`? What would happen to the average?
3. What would go wrong if you ran the load twice with a plain `INSERT` instead of an UPSERT?
4. Why does the database reject a ticket type that is not in `ticket_types`, and is that good or bad?
5. Why do we compute the closed rate with SQL instead of downloading everything into pandas?

In [17]:
# Recovery checkpoint: use this ONLY if your ETL was blocked.
# The instructor can run the same cleaning logic and load a reference snapshot.
RECOVER_FROM_CHECKPOINT = False
if RECOVER_FROM_CHECKPOINT:
    print("Ask the instructor to run the reference solution before continuing.")

print("Verified rows:", int(fetch_df(f"SELECT COUNT(*) AS n FROM {TEAM_SCHEMA}.clean_tickets").iloc[0,0]))

Verified rows: 600
